In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

df = pd.read_parquet("../data/processed/india_econ_trade.parquet")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


Shape: (769, 6)
Columns: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', 'Year', 'Value']


,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value
0,India,IND,"ICT service exports (BoP, current US$)",BX.GSR.CCIS.CD,2000,5.026789e+09
1,India,IND,"ICT service exports (BoP, current US$)",BX.GSR.CCIS.CD,2001,7.959226e+09
2,India,IND,"ICT service exports (BoP, current US$)",BX.GSR.CCIS.CD,2002,9.279017e+09
3,India,IND,"ICT service exports (BoP, current US$)",BX.GSR.CCIS.CD,2003,1.236021e+10
4,India,IND,"ICT service exports (BoP, current US$)",BX.GSR.CCIS.CD,2004,1.689115e+10


In [2]:
sectors = {
    "NV.AGR.TOTL.ZS": "Agriculture (% of GDP)",
    "NV.IND.MANF.ZS": "Manufacturing (% of GDP)",
    "NV.SRV.TOTL.ZS": "Services (% of GDP)"
}

df_sec = df[df["Indicator Code"].isin(sectors.keys())].copy()
df_sec["Sector"] = df_sec["Indicator Code"].map(sectors)

fig = px.line(
    df_sec,
    x="Year",
    y="Value",
    color="Sector",
    title="India's Structural Transformation: Share of GDP (1960–2025)",
    labels={"Value": "Share of GDP (%)", "Year": "Fiscal Year"},
    template="plotly_white"
)

# Add vertical line for 1991 Economic Liberalization
fig.add_vline(x=1991, line_dash="dash", line_color="gray", annotation_text="1991 Reforms")
fig.show()


In [3]:
def compare_eras(indicator_code, metric_name):
    sub = df[df["Indicator Code"] == indicator_code]
    pre_91 = sub[(sub["Year"] >= 1961) & (sub["Year"] <= 1990)]["Value"]
    post_91 = sub[(sub["Year"] >= 1991) & (sub["Year"] <= 2019)]["Value"]
    
    t_stat, p_val = stats.ttest_ind(post_91, pre_91, equal_var=False)
    print(f"--- {metric_name} ---")
    print(f"Pre-1991 Mean (1961-1990):  {pre_91.mean():.2f}%")
    print(f"Post-1991 Mean (1991-2019): {post_91.mean():.2f}%")
    print(f"Difference: +{post_91.mean() - pre_91.mean():.2f} percentage points (p = {p_val:.4e})\n")

compare_eras("NY.GNP.PCAP.KD.ZG", "Annual GNI Per Capita Growth (%)")
compare_eras("NE.TRD.GNFS.ZS", "Trade Openness (% of GDP)")


--- Annual GNI Per Capita Growth (%) ---
Pre-1991 Mean (1961-1990):  1.85%
Post-1991 Mean (1991-2019): 4.54%
Difference: +2.69 percentage points (p = 3.5935e-04)

--- Trade Openness (% of GDP) ---
Pre-1991 Mean (1961-1990):  11.60%
Post-1991 Mean (1991-2019): 36.01%
Difference: +24.41 percentage points (p = 2.7962e-11)



In [4]:
tech_codes = ["TX.VAL.TECH.MF.ZS", "TM.TAX.MRCH.WM.AR.ZS"]
df_tech = df[df["Indicator Code"].isin(tech_codes)].pivot(
    index="Year", columns="Indicator Code", values="Value"
).dropna()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=df_tech.index, y=df_tech["TX.VAL.TECH.MF.ZS"],
    name="High-Tech Exports (% of Mfg Exports)", mode="lines+markers"
))
fig2.add_trace(go.Scatter(
    x=df_tech.index, y=df_tech["TM.TAX.MRCH.WM.AR.ZS"],
    name="Weighted Mean Applied Tariff (%)", mode="lines+markers"
))
fig2.update_layout(
    title="India's Trade Modernization: High-Tech Exports vs. Import Tariffs",
    xaxis_title="Year", yaxis_title="Percentage (%)", template="plotly_white"
)
fig2.show()
